##Aim:
To build an end-to-end ETL pipeline for an e-commerce dataset that extracts CSV data, performs data cleaning and transformation, loads it into a SQL database, generates a report, and handles both historical and newly arriving data

In [ ]:
# ============================================================
# Practical No. 9: End-to-End E-Commerce ETL Pipeline
# ============================================================

import pandas as pd
import sqlite3


# ------------------------------------------------------------
# Step 1: Create E-Commerce CSV Files
# ------------------------------------------------------------

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "customer_name": ["Alice", "Bob", "Charlie", "David"],
    "city": ["Mumbai", "Delhi", "Pune", "Nashik"]
})

products = pd.DataFrame({
    "product_id": [101, 102, 103, 104],
    "product_name": ["Laptop", "Mobile", "Headphones", "Keyboard"],
    "category": ["Electronics", "Electronics", "Accessories", "Accessories"],
    "price": [50000, 20000, 3000, 1500]
})

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004],
    "customer_id": [1, 2, 3, 4],
    "product_id": [101, 102, 103, 104],
    "quantity": [1, 2, 2, 3]
})

payments = pd.DataFrame({
    "payment_id": [501, 502, 503, 504],
    "order_id": [1001, 1002, 1003, 1004],
    "payment_method": ["UPI", "Card", "UPI", "Cash"],
    "payment_status": ["Completed", "Completed", "Completed", "Pending"]
})

# Save datasets as CSV files
customers.to_csv("customers.csv", index=False)
products.to_csv("products.csv", index=False)
orders.to_csv("orders.csv", index=False)
payments.to_csv("payments.csv", index=False)


# ------------------------------------------------------------
# Step 2: Extract Data from CSV Files
# ------------------------------------------------------------

customers_df = pd.read_csv("customers.csv")
products_df = pd.read_csv("products.csv")
orders_df = pd.read_csv("orders.csv")
payments_df = pd.read_csv("payments.csv")

print("CSV data extracted successfully.")


# ------------------------------------------------------------
# Step 3: Data Cleaning
# ------------------------------------------------------------

# Remove duplicate records
customers_df.drop_duplicates(inplace=True)
products_df.drop_duplicates(inplace=True)
orders_df.drop_duplicates(inplace=True)
payments_df.drop_duplicates(inplace=True)

# Remove invalid product prices
products_df = products_df[products_df["price"] > 0]

# Remove invalid order quantities
orders_df = orders_df[orders_df["quantity"] > 0]

print("\nData cleaning completed.")


# ------------------------------------------------------------
# Step 4: Transform and Combine Data
# ------------------------------------------------------------

# Join orders with customer information
report_df = orders_df.merge(
    customers_df,
    on="customer_id",
    how="left"
)

# Join product information
report_df = report_df.merge(
    products_df,
    on="product_id",
    how="left"
)

# Join payment information
report_df = report_df.merge(
    payments_df,
    on="order_id",
    how="left"
)

# Calculate total order amount
report_df["total_amount"] = (
    report_df["quantity"] * report_df["price"]
)

print("\nTransformed E-Commerce Data:")
print(report_df)


# ------------------------------------------------------------
# Step 5: Load Data into SQL Database
# ------------------------------------------------------------

connection = sqlite3.connect("ecommerce.db")

# Store the final data in SQL table
report_df.to_sql(
    "ecommerce_orders",
    connection,
    if_exists="replace",
    index=False
)

print("\nData loaded into SQL database successfully.")


# ------------------------------------------------------------
# Step 6: Generate Report Using SQL
# ------------------------------------------------------------

report = pd.read_sql_query("""
    SELECT
        category,
        COUNT(order_id) AS total_orders,
        SUM(quantity) AS total_quantity,
        SUM(total_amount) AS total_sales
    FROM ecommerce_orders
    GROUP BY category
""", connection)

print("\n--- E-Commerce Sales Report ---")
print(report)


# ------------------------------------------------------------
# Step 7: Handle Newly Arriving Data
# ------------------------------------------------------------

new_orders = pd.DataFrame({
    "order_id": [1005],
    "customer_id": [1],
    "product_id": [102],
    "quantity": [1]
})

# Add new order to the existing orders
orders_df = pd.concat(
    [orders_df, new_orders],
    ignore_index=True
)

# Process the new order
new_order_report = new_orders.merge(
    customers_df,
    on="customer_id",
    how="left"
)

new_order_report = new_order_report.merge(
    products_df,
    on="product_id",
    how="left"
)

new_order_report["total_amount"] = (
    new_order_report["quantity"] *
    new_order_report["price"]
)

# Add default payment information
new_order_report["payment_id"] = 505
new_order_report["payment_method"] = "UPI"
new_order_report["payment_status"] = "Completed"

# Select same columns as database table
new_order_report = new_order_report[
    report_df.columns
]

# Insert newly arriving record into SQL database
new_order_report.to_sql(
    "ecommerce_orders",
    connection,
    if_exists="append",
    index=False
)

print("\nNewly arriving data loaded successfully.")


# ------------------------------------------------------------
# Step 8: Final Database Report
# ------------------------------------------------------------

final_report = pd.read_sql_query("""
    SELECT
        category,
        COUNT(order_id) AS total_orders,
        SUM(quantity) AS total_quantity,
        SUM(total_amount) AS total_sales
    FROM ecommerce_orders
    GROUP BY category
""", connection)

print("\n--- Final Report After New Data ---")
print(final_report)


# ------------------------------------------------------------
# Close Database Connection
# ------------------------------------------------------------

connection.close()

print("\nEnd-to-end ETL pipeline completed successfully!")


CSV data extracted successfully.

Data cleaning completed.

Transformed E-Commerce Data:
   order_id  customer_id  product_id  quantity customer_name    city  \
0      1001            1         101         1         Alice  Mumbai   
1      1002            2         102         2           Bob   Delhi   
2      1003            3         103         2       Charlie    Pune   
3      1004            4         104         3         David  Nashik   

  product_name     category  price  payment_id payment_method payment_status  \
0       Laptop  Electronics  50000         501            UPI      Completed   
1       Mobile  Electronics  20000         502           Card      Completed   
2   Headphones  Accessories   3000         503            UPI      Completed   
3     Keyboard  Accessories   1500         504           Cash        Pending   

   total_amount  
0         50000  
1         40000  
2          6000  
3          4500  

Data loaded into SQL database successfully.

--- E-Commerc

Urvi Sawant | 42